In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.cm as cm
from scipy.integrate import cumulative_trapezoid

from ptlpinns.models import model, transfer
from ptlpinns.perturbation import standard
from ptlpinns.odes import numerical, equations

plt.rcParams.update({
    "font.family": "serif",
    "font.serif": ["DejaVu Serif"],
    "mathtext.fontset": "cm",
    "text.usetex": False,
})

## Load pretrained model

In [ ]:
name = "overdamped_k15"
model_path = f"/home/jovyan/PTL-PINNs/ptlpinns/models/train/{name}"
model_file = f"model_{name}.pth"
pretrained_model, training_log = model.load_model(model_path, model_file)

## Problem definition

Overdamped Duffing-like oscillator:
$$\ddot{x} + 2\zeta\dot{x} + x + \varepsilon x^3 = \cos(t), \quad x(0)=1,\; \dot{x}(0)=0$$

We use $\zeta = 5$ (strongly overdamped), $\varepsilon = 0.5$, and compute the
perturbation series up to **first order** (corrections $x_0$ and $x_1$).

In [ ]:
N       = 150
t_span  = (0, 10)
t_eval  = np.linspace(t_span[0], t_span[1], N)

zeta       = 5.0
w_0        = 1.0
epsilon    = 0.5
ic         = [1.0, 0.0]
q          = [(3, 1)]      # cubic nonlinearity: epsilon * x^3

def forcing_np(t):
    return np.cos(t)

def forcing_2d(t):
    """Returns (N, 2) numpy array: [0, cos(t)] — 2-state layout expected by transfer."""
    return np.stack((np.zeros_like(t), np.cos(t)), axis=1)

## Build latent basis (H_dict)

In [ ]:
H_dict = transfer.compute_H_dict(pretrained_model, N=N, bias=True, t_span=t_span)

## Numerical reference (RK45)

In [ ]:
ode = equations.ode_oscillator_1D(w_0=w_0, zeta=zeta, forcing_1D=forcing_np, q=q, epsilon=epsilon)
numerical_sol = numerical.solve_ode_equation(ode, t_span, t_eval, ic)  # shape (2, N)

## PTL-PINN perturbation solution (orders 0 and 1)

In [ ]:
_, perturbation_solution, _ = transfer.compute_perturbation_solution(
    w_0_list     = [w_0],
    zeta_list    = [zeta],
    beta_list    = [epsilon],
    p_list       = [1],          # orders 0 and 1
    ic_list      = [ic],
    forcing_list = [forcing_2d],
    H_dict       = H_dict,
    t_eval       = t_eval,
    training_log = training_log,
    all_p        = True,
    comp_time    = False,
    solver       = "standard",
    power        = q,
)

# With a single zeta, the function returns perturbation_solution directly.
# perturbation_solution[i] is a (N, 2) array for correction order i.
x0 = perturbation_solution[0][:, 0]   # 0th-order correction
x1 = perturbation_solution[1][:, 0]   # 1st-order correction

series = standard.calculate_general_series([x0, x1], epsilon)
# series[0] = x0                 (order-0 approximation)
# series[1] = x0 + epsilon*x1   (order-1 approximation)

## Figure 1 — Individual correction terms $x_0$ and $x_1$

In [ ]:
LABEL_FS = 11
TICK_FS  = 9
TITLE_FS = 11

C_X0 = "#E57373"
C_X1 = "#F9A825"

fig, axes = plt.subplots(1, 2, figsize=(13, 3), sharey=False)

corrections = [(x0, r"$x_0(t)$", C_X0), (x1, r"$x_1(t)$", C_X1)]

for i, (ax, (xi, title, color)) in enumerate(zip(axes, corrections)):
    ax.plot(t_eval, xi, color=color, linewidth=3.5)
    ax.set_title(title, fontsize=TITLE_FS)
    ax.set_xlabel("$t$", fontsize=LABEL_FS)
    ax.tick_params(labelsize=TICK_FS)
    if i == 0:
        ax.set_ylabel("Amplitude", fontsize=LABEL_FS)

fig.suptitle(r"Perturbation corrections — overdamped oscillator ($\zeta=5$)",
             fontsize=TITLE_FS, y=1.02)
fig.tight_layout(pad=1.0, w_pad=1.2)
plt.savefig("overdamped_corrections.pdf", bbox_inches="tight", dpi=300)
plt.show()

## Figure 2 — Accumulated series vs RK45 (order 0 and order 1)

In [ ]:
C_X0  = "#E57373"
C_X1  = "#F9A825"
C_REF = "#546E7A"

fig, ax = plt.subplots(figsize=(7, 3.5))

ax.plot(t_eval, numerical_sol[0],
        label="RK45", color=C_REF, linewidth=2.0)
ax.plot(t_eval, series[0],
        label="PTL-PINN — order 0", color=C_X0,
        linewidth=1.6, linestyle="--")
ax.plot(t_eval, series[1],
        label="PTL-PINN — order 1", color=C_X1,
        linewidth=1.6, linestyle=":")

ax.set_xlabel("$t$", fontsize=LABEL_FS)
ax.set_ylabel("$x(t)$", fontsize=LABEL_FS)
ax.tick_params(labelsize=TICK_FS)
ax.legend(fontsize=TICK_FS, frameon=False, loc="upper right")

fig.tight_layout(pad=1.0)
plt.savefig("overdamped_series_vs_rk45.pdf", bbox_inches="tight", dpi=300)
plt.show()

## Figure 3 — Cumulative absolute error (IAE) for each order

In [ ]:
x_ref = numerical_sol[0]
C_X0  = "#E57373"
C_X1  = "#F9A825"

fig, ax = plt.subplots(figsize=(7, 3))

for i, (sol, label, color) in enumerate([
    (series[0], "Order 0", C_X0),
    (series[1], "Order 1", C_X1),
]):
    iae = cumulative_trapezoid(np.abs(sol - x_ref), t_eval, initial=0)
    ax.plot(t_eval, iae, label=label, color=color,
            linewidth=1.6, linestyle="--" if i == 0 else ":")

ax.set_xlabel("$t$", fontsize=LABEL_FS)
ax.set_ylabel("IAE$(t)$", fontsize=LABEL_FS)
ax.tick_params(labelsize=TICK_FS)
ax.legend(fontsize=TICK_FS, frameon=False)

fig.tight_layout(pad=1.0)
plt.savefig("overdamped_iae.pdf", bbox_inches="tight", dpi=300)
plt.show()

## MAE per order

In [ ]:
for i, sol in enumerate(series):
    mae = np.mean(np.abs(sol - x_ref))
    print(f"Order {i} — MAE: {mae:.4e}")